## Phase 3 — finetuning for reasoning (final run)

**Run this with `Save Version → Save & Run All (Commit)`, not interactively.**

Accelerator `GPU T4 x2`, Internet `Off`. Attach `lma-phase3-code`, `lma-phase3-ckpt`, `lma-phase3-ckpt-mr`.

There is nothing to set by hand. Every configuration choice is fixed in cell 1 with the evidence beside it. Expected runtime about 100 minutes; results are written to disk after each experiment, so a failure late in the run does not cost the earlier ones.


In [ ]:
# =============================================================================
# PHASE 3 - FINAL RUN
#
# Accelerator: GPU T4 x2.  Internet: OFF.
# Attach: lma-phase3-code, lma-phase3-ckpt, lma-phase3-ckpt-mr
#
# RUN THIS WITH "Save Version -> Save & Run All (Commit)", NOT INTERACTIVELY.
#
# There is nothing to set by hand anywhere in this notebook. Every choice was
# already decided by the runs of 14 September and is hardcoded below with the
# evidence next to it. Start it, close the tab, come back.
#
# WHAT THE EARLIER RUNS ESTABLISHED
# ---------------------------------
#   learning rate   1e-4 destroys the pretrained model even at one epoch
#                   (Marathi perplexity x470, Konkani x81). 5e-6 leaves it
#                   intact (x1.08-x1.15). 2e-5 is halfway and buys nothing.
#
#   target          Answer-only beats chain-of-thought at every learning rate
#                   and every epoch count, in both languages. The chain gets
#                   worse with training (Marathi 9.4 -> 10.8 -> 5.3) and emits
#                   entity names that are not among the options at all.
#
#   epochs          Konkani answer-only rises monotonically: 24.20 (1), 25.10
#                   (3), 28.20 (6). Six is the only setting anywhere above the
#                   chance floor. Twelve is tested here to see if it continues.
#
# WHAT IS BEING MEASURED
# ----------------------
#   A  recipe        learning rate x target, 1 epoch, N=8000        12 runs
#   B  epochs        1/3/6/12 at the chosen recipe                   6 new runs
#   C  sample count  500..8000 at the chosen recipe                  8 new runs
#   D  final         the chosen model against its own pretrained checkpoint
#   E  diagnostic    what the model actually emits, and both floors
#   F  attention     pretrained vs finetuned, specification 3.2
#
# Runs are reused across experiments rather than repeated, so this is 26
# training runs and 30 evaluations, roughly 100 minutes.
# =============================================================================


# Inputs, and every decision this notebook makes.

import os, glob, json, csv, math, shutil, subprocess, sys, time
from collections import Counter

T0 = time.time()

def find(marker):
    for root, dirs, files in os.walk('/kaggle/input'):
        if marker in dirs or marker in files:
            return os.path.join(root, marker)
    return None

CODE = os.path.dirname(find('tools'))
PRETRAINED = {lang: find(f'{lang}_pretrain_best.pt')
              for lang in ['marathi', 'konkani']}
LANGS = ['marathi', 'konkani']
REPO, OUT = '/kaggle/working/repo', '/kaggle/working/out'

# ---- the decisions, fixed -------------------------------------------------
FINAL_LR        = 5e-6        # the only rate that preserves the language model
FINAL_VARIANT   = 'plain'     # answer-only; the chain-of-thought target is worse
FINAL_RATIONALE = False
FINAL_EPOCHS    = 6           # best mean lift, and the only row above chance
FINAL_N         = 8000

GRID_LRS      = [('5e6', 5e-6), ('2e5', 2e-5), ('1e4', 1e-4)]
GRID_VARIANTS = [('plain', False), ('rationale', True)]
EPOCH_POINTS  = [3, 6, 12]                  # 1 comes from the recipe grid
SIZE_POINTS   = [500, 1000, 2000, 4000]     # 8000 comes from the epoch sweep
FORGET_BUDGET = 1.5           # reject a config whose pretraining ppl rose >50%

assert CODE, 'lma-phase3-code is not attached'
assert all(PRETRAINED.values()), f'missing checkpoint: {PRETRAINED}'
print('CODE =', CODE)
for lang, p in PRETRAINED.items():
    print(f'  {lang:9} {p}  ({os.path.getsize(p)/1e6:.0f} MB)')

# The two checkpoints sit in different datasets, because the first upload failed
# partway and Marathi had to be re-uploaded on its own. Each is therefore found
# by filename rather than by folder, which is only safe if the file really is
# the model its name claims. Every checkpoint records the language it was
# trained on inside train_config, so check that rather than trusting the name:
# a Marathi model finetuned on Konkani data would still run, and would quietly
# produce nonsense.
import torch
for lang, p in PRETRAINED.items():
    blob = torch.load(p, map_location='cpu', weights_only=False)
    declared = blob['train_config']['language']
    assert declared == lang, f'{p} holds a {declared} model, not {lang}'
    print(f'  {lang:9} declares language={declared}, {blob["step"]:,} steps, '
          f"val loss {blob.get('best_val', float('nan')):.4f}  OK")
    del blob
print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)


In [ ]:
# Copy the code somewhere writable and check it before spending GPU time.

if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(CODE, REPO)
os.makedirs(f'{OUT}/evals', exist_ok=True)
os.makedirs(f'{OUT}/report', exist_ok=True)

import sentencepiece
print('sentencepiece', sentencepiece.__version__)
r = subprocess.run([sys.executable, 'tools/verify_model.py'],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-600:]); print(r.stderr[-300:])
assert 'checks passed' in r.stdout, 'model verification failed'

for lang in LANGS:
    n = sum(1 for _ in open(f'{REPO}/{lang}/data/reasoning/train.jsonl'))
    t = sum(1 for _ in open(f'{REPO}/{lang}/data/reasoning/test.jsonl'))
    print(f'{lang}: {n} train, {t} test reasoning items')


In [ ]:
# Helpers, and the pretrained baseline every later number is measured against.

RESULTS = []          # every (language, tag) measurement, appended as it lands

def save_results():
    """Written after each experiment, so a crash never costs the whole run."""
    with open(f'{OUT}/report/phase3_results.json', 'w') as fh:
        json.dump({'results': RESULTS, 'baseline': baseline,
                   'final_config': {'lr': FINAL_LR, 'variant': FINAL_VARIANT,
                                    'epochs': FINAL_EPOCHS, 'n_train': FINAL_N}},
                  fh, indent=2, ensure_ascii=False)
    if RESULTS:
        keys = sorted({k for r in RESULTS for k in r})
        with open(f'{OUT}/report/phase3_results.csv', 'w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=keys); w.writeheader()
            w.writerows(RESULTS)

def prune_latest():
    freed = 0
    for p in glob.glob(f'{OUT}/**/*_latest.pt', recursive=True):
        freed += os.path.getsize(p); os.remove(p)
    if freed:
        print(f'  pruned {freed/1e9:.2f} GB of resumable checkpoints', flush=True)

def prune_runs(pattern, keep=()):
    """Delete scored run directories. Kaggle allows 19.5 GB and each run is
    ~285 MB; the numbers are already in RESULTS by the time this is called."""
    freed = 0
    for d in glob.glob(pattern):
        if os.path.basename(d) in keep:
            continue
        for root, _, files in os.walk(d):
            for f in files:
                freed += os.path.getsize(os.path.join(root, f))
        shutil.rmtree(d)
    if freed:
        print(f'  cleared {freed/1e9:.2f} GB', flush=True)

def run_parallel(jobs, poll=30):
    procs = []
    for name, cmd, gpu in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1')
        log = open(f'/kaggle/working/{name}.log', 'w')
        procs.append((name, subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log,
                                             stderr=subprocess.STDOUT), log))
    started = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(poll)
        alive = [n for n, p, _ in procs if p.poll() is None]
        print(f'    [{time.time()-started:5.0f}s] {", ".join(alive)}', flush=True)
    ok = True
    for name, p, log in procs:
        log.close()
        if p.returncode != 0:
            ok = False
            print(f'  {name} FAILED ({p.returncode}):')
            print(open(f'/kaggle/working/{name}.log').read()[-1500:])
        else:
            tail = [l for l in open(f'/kaggle/working/{name}.log').read().splitlines()
                    if 'best val loss' in l]
            print(f'  {name} ok  {tail[-1].strip() if tail else ""}', flush=True)
    return ok

def train(tag, lr, epochs, n_train, rationale):
    """One training run per language, one per GPU, in parallel."""
    print(f'===== train {tag}  lr={lr:g} epochs={epochs} N={n_train} '
          f"target={'chain' if rationale else 'answer-only'} =====", flush=True)
    jobs = []
    for gpu, lang in enumerate(LANGS):
        cmd = [sys.executable, 'tools/finetune.py', '--language', lang,
               '--pretrained', PRETRAINED[lang],
               '--data-dir', f'{REPO}/{lang}/data/reasoning',
               '--out-dir', f'{OUT}/{lang}/{tag}',
               '--n-train', str(n_train), '--epochs', str(epochs),
               '--lr', str(lr), '--tag', tag, '--device', 'cuda']
        if rationale:
            cmd.append('--rationale')
        jobs.append((f'{tag}_{lang}', cmd, gpu))
    assert run_parallel(jobs), f'{tag} failed'
    prune_latest()

def measure(lang, tag, rationale, ckpt=None, windows=128, compare=False, **meta):
    """Evaluate one checkpoint and append the summary to RESULTS."""
    ckpt = ckpt or f'{OUT}/{lang}/{tag}/finetune_{tag}_best.pt'
    cmd = [sys.executable, 'tools/evaluate_reasoning.py',
           '--language', lang, '--checkpoint', ckpt,
           '--data-dir', f'{REPO}/{lang}/data/reasoning',
           '--lm-data-dir', f'{REPO}/{lang}/data/packed',
           '--lm-windows', str(windows),
           '--out-dir', f'{OUT}/evals', '--tag', tag,
           '--max-new-tokens', '64' if rationale else '24',
           '--batch-size', '64', '--device', 'cuda']
    if compare:
        cmd += ['--compare-to', PRETRAINED[lang]]
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    path = f'{OUT}/evals/phase3_reasoning_eval_{lang}_{tag}.json'
    if not os.path.exists(path):
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
        raise RuntimeError(f'evaluation failed: {lang} {tag}')
    f = json.load(open(path))['finetuned']
    row = {'language': lang, 'tag': tag,
           'accuracy': f['overall']['accuracy'],
           'chance': f['overall']['chance_accuracy'],
           'format': f['overall']['format_compliance'],
           'lm_ppl': f['language_model']['perplexity'],
           'lm_bpb': f['language_model']['bits_per_byte']}
    row.update(meta)
    if 'baseline' in globals() and lang in baseline:
        row['ppl_ratio'] = round(row['lm_ppl'] / baseline[lang]['lm_ppl'], 4)
        row['lift'] = round(row['accuracy'] - row['chance'], 2)
    RESULTS.append(row)
    print(f"  {lang:9} {tag:<18} acc {row['accuracy']:6.2f}%  "
          f"format {row['format']:6.2f}%  LM ppl {row['lm_ppl']:9.2f}"
          + (f"  x{row['ppl_ratio']:.2f}" if 'ppl_ratio' in row else ''),
          flush=True)
    if compare:
        print(r.stdout[-1800:])
    return row

# ---- the reference point --------------------------------------------------
baseline = {}
for lang in LANGS:
    r = subprocess.run([
        sys.executable, 'tools/evaluate_reasoning.py', '--language', lang,
        '--checkpoint', PRETRAINED[lang],
        '--data-dir', f'{REPO}/{lang}/data/reasoning',
        '--lm-data-dir', f'{REPO}/{lang}/data/packed', '--lm-windows', '128',
        '--out-dir', f'{OUT}/evals', '--tag', 'pretrained',
        '--max-new-tokens', '24', '--batch-size', '64', '--device', 'cuda',
    ], cwd=REPO, capture_output=True, text=True)
    f = json.load(open(
        f'{OUT}/evals/phase3_reasoning_eval_{lang}_pretrained.json'))['finetuned']
    baseline[lang] = {'accuracy': f['overall']['accuracy'],
                      'chance': f['overall']['chance_accuracy'],
                      'format': f['overall']['format_compliance'],
                      'lm_ppl': f['language_model']['perplexity'],
                      'lm_bpb': f['language_model']['bits_per_byte']}
    b = baseline[lang]
    print(f"{lang:9} PRETRAINED  acc {b['accuracy']:5.2f}%  chance {b['chance']:5.2f}%"
          f"  format {b['format']:5.2f}%  LM ppl {b['lm_ppl']:8.2f}  "
          f"bpb {b['lm_bpb']:.4f}", flush=True)
# Second check on the pairing: these are the same models Phase 2 reported, so
# their perplexity must land on the Phase 2 figures. The small gap is the window
# count (128 here against 256 there). A mismatched checkpoint and tokenizer
# would be off by orders of magnitude, not by one per cent.
PHASE2_PPL = {'marathi': 8.62, 'konkani': 26.56}
for lang in LANGS:
    got, want = baseline[lang]['lm_ppl'], PHASE2_PPL[lang]
    assert abs(got - want) / want < 0.15, (
        f'{lang} pretrained perplexity {got:.2f} is nowhere near the Phase 2 '
        f'figure {want:.2f} - wrong checkpoint, tokenizer or data split')
    print(f'  {lang:9} agrees with Phase 2: {got:.2f} vs {want:.2f}')

save_results()
print(f'\n[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# EXPERIMENT A - the recipe. Learning rate x training target, 1 epoch, N=8000.

for vname, rat in GRID_VARIANTS:
    for ltag, lr in GRID_LRS:
        train(f'grid_{vname}_{ltag}', lr, 1, 8000, rat)

for vname, rat in GRID_VARIANTS:
    for ltag, lr in GRID_LRS:
        for lang in LANGS:
            measure(lang, f'grid_{vname}_{ltag}', rat, experiment='recipe',
                    variant=vname, lr=lr, epochs=1, n_train=8000)
save_results()
# keep only the row this notebook builds on: answer-only at 5e-6, 1 epoch
prune_runs(f'{OUT}/*/grid_*', keep=())

print('\n language  variant     lr        acc%  chance%    lift  format%     '
      'LM ppl   ppl x   verdict')
for lang in LANGS:
    b = baseline[lang]
    print(f"  {lang:<9}{'PRETRAINED':<12}{'-':<9}{b['accuracy']:6.2f} "
          f"{b['chance']:8.2f} {b['accuracy']-b['chance']:+7.2f} {b['format']:8.2f} "
          f"{b['lm_ppl']:10.2f}       -")
    for r in [x for x in RESULTS if x.get('experiment') == 'recipe'
              and x['language'] == lang]:
        v = 'LM LOST' if r['ppl_ratio'] > FORGET_BUDGET else (
            'above chance' if r['lift'] > 0 else '')
        print(f"  {lang:<9}{r['variant']:<12}{r['lr']:<9.0e}{r['accuracy']:6.2f} "
              f"{r['chance']:8.2f} {r['lift']:+7.2f} {r['format']:8.2f} "
              f"{r['lm_ppl']:10.2f}  x{r['ppl_ratio']:6.2f}   {v}")
print(f'\n[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# EXPERIMENT B - how long to train, at the chosen recipe.
# The 1-epoch point is the grid run, reused rather than repeated.

for ep in EPOCH_POINTS:
    train(f'epochs_e{ep}', FINAL_LR, ep, FINAL_N, FINAL_RATIONALE)
for ep in EPOCH_POINTS:
    for lang in LANGS:
        measure(lang, f'epochs_e{ep}', FINAL_RATIONALE, experiment='epochs',
                variant=FINAL_VARIANT, lr=FINAL_LR, epochs=ep, n_train=FINAL_N)
save_results()
prune_runs(f'{OUT}/*/epochs_*', keep=(f'epochs_e{FINAL_EPOCHS}',))

print('\n language  epochs   acc%  chance%    lift  format%     LM ppl   ppl x')
for lang in LANGS:
    rows = ([x for x in RESULTS if x['language'] == lang
             and x['tag'] == f'grid_{FINAL_VARIANT}_5e6'] +
            [x for x in RESULTS if x['language'] == lang
             and x.get('experiment') == 'epochs'])
    for r in sorted(rows, key=lambda r: r['epochs']):
        print(f"  {lang:<9}{r['epochs']:<9}{r['accuracy']:6.2f} {r['chance']:8.2f} "
              f"{r['lift']:+7.2f} {r['format']:8.2f} {r['lm_ppl']:10.2f}  "
              f"x{r['ppl_ratio']:6.2f}")
print(f'\n[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# EXPERIMENT C - how much data, at the chosen recipe and epoch count.
# N=8000 is the epoch-sweep run, reused.

for n in SIZE_POINTS:
    train(f'n{n}', FINAL_LR, FINAL_EPOCHS, n, FINAL_RATIONALE)
for n in SIZE_POINTS:
    for lang in LANGS:
        measure(lang, f'n{n}', FINAL_RATIONALE, experiment='samples',
                variant=FINAL_VARIANT, lr=FINAL_LR, epochs=FINAL_EPOCHS,
                n_train=n)
# relabel the reused N=8000 run so the sweep is complete
for lang in LANGS:
    src = [x for x in RESULTS if x['language'] == lang
           and x['tag'] == f'epochs_e{FINAL_EPOCHS}'][0]
    RESULTS.append({**src, 'tag': 'n8000', 'experiment': 'samples',
                    'n_train': FINAL_N})
save_results()
prune_runs(f'{OUT}/*/n*', keep=())

sweep = sorted([x for x in RESULTS if x.get('experiment') == 'samples'],
               key=lambda r: (r['language'], r['n_train']))
print('\n language  N        acc%  chance%    lift  format%     LM ppl   ppl x')
for r in sweep:
    print(f"  {r['language']:<9}{r['n_train']:<9}{r['accuracy']:6.2f} "
          f"{r['chance']:8.2f} {r['lift']:+7.2f} {r['format']:8.2f} "
          f"{r['lm_ppl']:10.2f}  x{r['ppl_ratio']:6.2f}")
print(f'\n[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# The three figures. Colours match the Phase 2 training curves.

# Default backend, not Agg: the figures should render inside the committed
# notebook as well as being written to disk.
import matplotlib.pyplot as plt

C = {'marathi': '#1f77b4', 'konkani': '#d62728'}
def tag_of(lang):
    return f"{lang.capitalize()} (Model {'H' if lang == 'marathi' else 'L'})"

# ---- figure 1: sample count ------------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
for lang in LANGS:
    s = [r for r in sweep if r['language'] == lang]
    xs = [r['n_train'] for r in s]
    a1.plot(xs, [r['accuracy'] for r in s], marker='o', color=C[lang],
            label=tag_of(lang))
    a1.plot(xs, [r['chance'] for r in s], ls=':', color=C[lang],
            label=f'{tag_of(lang)} — chance')
    a1.plot(xs, [r['format'] for r in s], marker='s', ms=4, ls='-.',
            color=C[lang], alpha=.5, label=f'{tag_of(lang)} — format compliance')
    a2.plot(xs, [r['lm_ppl'] for r in s], marker='o', color=C[lang],
            label=tag_of(lang))
    a2.axhline(baseline[lang]['lm_ppl'], ls='--', lw=1, color=C[lang], alpha=.6,
               label=f'{tag_of(lang)} — pretrained')
a1.set_title('Reasoning accuracy and format compliance')
a1.set_xlabel('finetuning samples'); a1.set_ylabel('per cent')
a1.set_xscale('log'); a1.legend(fontsize=7); a1.grid(alpha=.3, which='both')
a2.set_title('Cost: perplexity on the pretraining test split')
a2.set_xlabel('finetuning samples'); a2.set_ylabel('perplexity')
a2.set_xscale('log'); a2.legend(fontsize=8); a2.grid(alpha=.3, which='both')
fig.suptitle(f'Finetuning sample count — answer-only target, lr={FINAL_LR:g}, '
             f'{FINAL_EPOCHS} epochs')
fig.tight_layout(); fig.savefig(f'{OUT}/report/phase3_samples.png', dpi=140)
plt.show()

# ---- figure 2: epochs ------------------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
for lang in LANGS:
    rows = ([x for x in RESULTS if x['language'] == lang
             and x['tag'] == f'grid_{FINAL_VARIANT}_5e6'] +
            [x for x in RESULTS if x['language'] == lang
             and x.get('experiment') == 'epochs'])
    rows = sorted(rows, key=lambda r: r['epochs'])
    xs = [r['epochs'] for r in rows]
    a1.plot(xs, [r['accuracy'] for r in rows], marker='o', color=C[lang],
            label=tag_of(lang))
    a1.plot(xs, [r['chance'] for r in rows], ls=':', color=C[lang],
            label=f'{tag_of(lang)} — chance')
    a2.plot(xs, [r['lm_ppl'] for r in rows], marker='o', color=C[lang],
            label=tag_of(lang))
    a2.axhline(baseline[lang]['lm_ppl'], ls='--', lw=1, color=C[lang], alpha=.6,
               label=f'{tag_of(lang)} — pretrained')
a1.set_title('Reasoning accuracy against the chance floor')
a1.set_xlabel('epochs'); a1.set_ylabel('exact-match accuracy (%)')
a1.legend(fontsize=8); a1.grid(alpha=.3)
a2.set_title('Cost: perplexity on the pretraining test split')
a2.set_xlabel('epochs'); a2.set_ylabel('perplexity')
a2.legend(fontsize=8); a2.grid(alpha=.3)
fig.suptitle(f'Training length — answer-only target, lr={FINAL_LR:g}, N={FINAL_N}')
fig.tight_layout(); fig.savefig(f'{OUT}/report/phase3_epochs.png', dpi=140)
plt.show()

# ---- figure 3: the learning-rate cliff -------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 5))
for lang in LANGS:
    for vname, _ in GRID_VARIANTS:
        rows = sorted([x for x in RESULTS if x.get('experiment') == 'recipe'
                       and x['language'] == lang and x['variant'] == vname],
                      key=lambda r: r['lr'])
        ax.plot([r['lr'] for r in rows], [r['ppl_ratio'] for r in rows],
                marker='o' if vname == 'plain' else '^',
                ls='-' if vname == 'plain' else '--', color=C[lang],
                label=f'{tag_of(lang)} — {vname}')
ax.axhline(FORGET_BUDGET, color='k', lw=1, ls=':',
           label=f'forgetting budget (x{FORGET_BUDGET})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('learning rate'); ax.set_ylabel('perplexity relative to pretrained')
ax.set_title('Catastrophic forgetting against learning rate\n(1 epoch, N=8000)')
ax.legend(fontsize=8); ax.grid(alpha=.3, which='both')
fig.tight_layout(); fig.savefig(f'{OUT}/report/phase3_forgetting.png', dpi=140)
plt.show()
print(f'[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# EXPERIMENT D - the final models, measured against their own pretrained
# checkpoints at the Phase 2 window count, with the delta tables the report needs.

FINAL_TAG = f'epochs_e{FINAL_EPOCHS}'
final_ckpt = {lang: f'{OUT}/{lang}/{FINAL_TAG}/finetune_{FINAL_TAG}_best.pt'
              for lang in LANGS}
for lang in LANGS:
    print(f'########## {lang} — final ##########')
    measure(lang, 'final', FINAL_RATIONALE, ckpt=final_ckpt[lang], windows=256,
            compare=True, experiment='final', variant=FINAL_VARIANT,
            lr=FINAL_LR, epochs=FINAL_EPOCHS, n_train=FINAL_N)
save_results()


In [ ]:
# EXPERIMENT E - what the model actually emits, and the two floors it has to
# clear. No GPU: this reads the evaluation JSONs already on disk.

def collapse(lang, tag, label):
    path = f'{OUT}/evals/phase3_reasoning_eval_{lang}_{tag}.json'
    if not os.path.exists(path):
        return None
    rows = json.load(open(path))['finetuned']['rows']
    ans = [r for r in rows if r['emitted_marker']]
    print(f'\n=== {lang} / {label} ===')
    print(f'    {len(ans)}/{len(rows)} items produced a parseable answer')
    print('    family                n   acc%  uniform%  majority%   '
          'top prediction        share   preds/golds')
    out = {}
    for fam in sorted({r['family'] for r in rows}):
        sub = [r for r in rows if r['family'] == fam]
        sa = [r for r in sub if r['emitted_marker']]
        if not sa:
            print(f'    {fam:<20}{len(sub):>4}   no parseable answers'); continue
        preds, golds = Counter(r['predicted'] for r in sa), Counter(r['gold'] for r in sub)
        top, cnt = preds.most_common(1)[0]
        acc = 100 * sum(r['correct'] for r in sub) / len(sub)
        uni = 100 * sum(r['chance'] for r in sub) / len(sub)
        maj = 100 * golds.most_common(1)[0][1] / len(sub)
        print(f'    {fam:<20}{len(sub):>4}  {acc:5.2f}  {uni:8.2f}  {maj:9.2f}   '
              f'{top!r:<20} {100*cnt/len(sa):5.1f}%   {len(preds):>3}/{len(golds):<3}')
        out[fam] = {'n': len(sub), 'accuracy': round(acc, 2),
                    'uniform_chance': round(uni, 2), 'majority_baseline': round(maj, 2),
                    'top_prediction': top,
                    'top_prediction_share': round(100 * cnt / len(sa), 2),
                    'distinct_predictions': len(preds), 'distinct_golds': len(golds)}
    return out

def floors(lang, tag):
    """Per-item Bernoulli test: chance is 1/(k+1) per item, so the variance is
    the sum of p(1-p), not n*p(1-p) at one p. One-tailed."""
    path = f'{OUT}/evals/phase3_reasoning_eval_{lang}_{tag}.json'
    if not os.path.exists(path):
        return None
    rows = json.load(open(path))['finetuned']['rows']
    n, k = len(rows), sum(r['correct'] for r in rows)
    ps = [r['chance'] for r in rows]
    mu, var = sum(ps), sum(p * (1 - p) for p in ps)
    z = (k - mu) / math.sqrt(var) if var > 0 else 0.0
    maj = sum(Counter(r['gold'] for r in rows if r['family'] == fam).most_common(1)[0][1]
              for fam in {r['family'] for r in rows})
    return {'n': n, 'correct': k, 'accuracy': round(100 * k / n, 2),
            'uniform': round(100 * mu / n, 2), 'majority': round(100 * maj / n, 2),
            'z': round(z, 3), 'p_one_tailed': round(0.5 * math.erfc(z / math.sqrt(2)), 4)}

DIAG_TAGS = [('final', f'answer-only, lr={FINAL_LR:g}, {FINAL_EPOCHS} epochs'),
             ('grid_plain_5e6', 'answer-only, lr=5e-6, 1 epoch'),
             ('grid_rationale_5e6', 'chain of thought, lr=5e-6, 1 epoch'),
             ('grid_plain_1e4', 'answer-only, lr=1e-4 (language model destroyed)'),
             ('pretrained', 'pretrained, no finetuning')]

diagnostic, verdicts = {}, {}
for lang in LANGS:
    for tag, label in DIAG_TAGS:
        d = collapse(lang, tag, label)
        if d:
            diagnostic[f'{lang}/{tag}'] = d
        f = floors(lang, tag)
        if f:
            verdicts[f'{lang}/{tag}'] = f

print('\n language  config                                    acc%  uniform%  '
      'majority%      z       p   clears')
for lang in LANGS:
    for tag, label in DIAG_TAGS:
        f = verdicts.get(f'{lang}/{tag}')
        if not f:
            continue
        cl = [name for name, v in (('uniform', f['uniform']), ('majority', f['majority']))
              if f['accuracy'] > v]
        print(f"  {lang:<9}{label:<42}{f['accuracy']:6.2f}  {f['uniform']:8.2f}  "
              f"{f['majority']:9.2f}  {f['z']:+6.2f}  {f['p_one_tailed']:6.3f}   "
              f"{', '.join(cl) if cl else 'neither'}")

json.dump(diagnostic, open(f'{OUT}/report/phase3_collapse.json', 'w'),
          indent=2, ensure_ascii=False)
json.dump(verdicts, open(f'{OUT}/report/phase3_floors.json', 'w'), indent=2)
print("\n  uniform%  answer at random among the entities offered")
print("  majority% answer each family's commonest label, ignoring the input")
print("  p < 0.05 is the bar for claiming the model beats guessing")
print(f'\n[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# EXPERIMENT F - attention, pretrained against finetuned. Specification 3.2.
# Wrapped, because a failure here must not cost the results above.

for lang in LANGS:
    for label, ck in [('pretrained', PRETRAINED[lang]),
                      ('finetuned', final_ckpt[lang])]:
        try:
            r = subprocess.run([
                sys.executable, 'tools/attention_analysis.py',
                '--language', lang, '--checkpoint', ck,
                '--tokenizer', f'{REPO}/{lang}/tokenizer/{lang}_bpe.model',
                '--data-dir', f'{REPO}/{lang}/data/packed',
                '--split', 'test', '--device', 'cuda',
            ], cwd=REPO, capture_output=True, text=True, timeout=1800)
            print(f'--- {lang} / {label} ---')
            print(r.stdout[-1400:] or r.stderr[-900:], flush=True)
            for pat in (f'{REPO}/report/phase2_attention_{lang}*',
                        f'{REPO}/report/figures/phase2_attention_{lang}*'):
                for f in glob.glob(pat):
                    shutil.move(f, f'{OUT}/report/'
                                   f'attn_{label}_{os.path.basename(f)}')
        except Exception as e:
            print(f'attention analysis failed for {lang}/{label}: {e}')
print(f'[{time.time()-T0:.0f}s elapsed]')


In [ ]:
# Package. Two zips in the Output panel: results (small) and final checkpoints.

import zipfile
for lang in LANGS:
    src = f'{OUT}/{lang}/{FINAL_TAG}/finetune_{FINAL_TAG}_log.csv'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/report/{lang}_finetune_log.csv')

with zipfile.ZipFile('/kaggle/working/phase3_results.zip', 'w',
                     zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(f'{OUT}/report'):
        for f in files:
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, f'{OUT}/report'))
    for f in glob.glob(f'{OUT}/evals/*.json'):
        z.write(f, 'evals/' + os.path.basename(f))
    for f in glob.glob('/kaggle/working/*.log'):
        z.write(f, 'logs/' + os.path.basename(f))
print('results    ', round(os.path.getsize('/kaggle/working/phase3_results.zip')/1e6, 1), 'MB')

with zipfile.ZipFile('/kaggle/working/phase3_checkpoints.zip', 'w',
                     zipfile.ZIP_STORED) as z:
    for lang in LANGS:
        z.write(final_ckpt[lang], f'{lang}_finetune_final_best.pt')
print('checkpoints', round(os.path.getsize('/kaggle/working/phase3_checkpoints.zip')/1e6, 1), 'MB')
print(f'\nTOTAL RUNTIME {(time.time()-T0)/60:.1f} minutes')
print('Download both zips from the Output panel of this version.')
